# *This notebook provides a dedicated workflow to process each annnotation-based database considered from raw data to a network file*.

### *Importing the required libraries*

In [1]:
import sys
sys.path.append("../scripts")

import glob
import json
import omics_analysis
import os
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
import shutil
import useful_functions

from datetime import datetime
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from sklearn.model_selection import KFold
from tqdm import tqdm

### *Reading the training genes*

In [ ]:
genes = pd.read_csv("../TrainingGenes/training_genes_stalk_cell.csv")
genes = genes["Feature"].to_list()

### *Processing BioGrid*

In [ ]:
import biogrid_analysis as BGA
import preprocess_biogrid as pb

# Setting the process of interest
process = "stalk_cell"

# Setting the path for the network that will be generated
annot_network_dir = "../databases"

# Setting the path for the graph that will be generated
annot_graph_dir = "../graphs/databases"

# Creating the network and graph files only if they don't already exist
if not os.path.exists(f"{annot_network_dir}/biogrid_network.csv"):

# Reading the initial raw data and applying preprocessing
    biogrid = pd.read_csv(f"{annot_network_dir}/BIOGRID-ALL-4.4.246.tab3.zip",sep = "\t")
    biogrid = biogrid[(biogrid["Organism ID Interactor A"] == 9606)]
    set(biogrid["Experimental System"].to_list())
    biogrid[biogrid["Experimental System"] == "Synthetic Lethality"]["Experimental System Type"].unique()

# Keeping data from trustworthy results only
    trustworthy_methods = [
        'Affinity Capture-MS', 'Co-crystal Structure', 
        'Cross-Linking-MS (XL-MS)', 'Proximity Label-MS',
        'Co-purification','Two-hybrid', 
        'Synthetic Lethality',
        'Proximity Label-MS',
        'FRET'
    ]
    biogrid_filtered = biogrid[
        (biogrid['Experimental System'].isin(trustworthy_methods)) &
        (biogrid['Experimental System Type'].isin(['physical' , 'genetic']) )
    ]
    methods = set(biogrid_filtered["Experimental System"].to_list())

# Checking how much data has been kept after filtering
    print(biogrid_filtered["Experimental System"].value_counts())
    print(biogrid_filtered[biogrid_filtered["Score"] == "-"]["Experimental System"].value_counts())

# Filtering the scores in the dataset
    biogrid_filtered["Score"] = biogrid_filtered["Score"].replace("-", np.nan)
    cols = ["Official Symbol Interactor A", "Official Symbol Interactor B", "Score", "Experimental System"]
    biogrid_filtered = biogrid_filtered[cols]
    biogrid_filtered[~biogrid_filtered["Score"].isna()].head()

# Assessing whether the methods are numeric or not
    method_type = {}
    for method in methods:
        scores = biogrid_filtered[biogrid_filtered["Experimental System"] == method]["Score"]
        if sum(scores.isna()) == len(scores):
            method_type[method] = "not_numeric"
        else:
            method_type[method] = "numeric"
    print("Summary of the methods types\n", method_type)

    numeric_methods = [key for key, value in method_type.items() if value == "numeric"]
    not_numeric_methods = [key for key, value in method_type.items() if value == "not_numeric"]
    print("NUMERIC METHODS\n", numeric_methods)
    print("NOT NUMERIC METHODS\n", not_numeric_methods)

# Defining the groups
    source_systems = numeric_methods
    target_systems = not_numeric_methods

# Filtering invalid and NaN scores
    biogrid_filtered = pb.filter_invalid_and_nan_scores(biogrid_filtered, 
                                                        source_systems)

# Normalizing the scores
    biogrid_filtered = pb.normalize_scores(biogrid_filtered, 
                                           source_systems)

# Imputing scores for not-numeric systems
    biogrid_filtered = pb.assign_most_frequent_scores(biogrid_filtered,
                                                      source_systems,
                                                      target_systems)

# log10 normalizing the scores
    biogrid_filtered["Normalized Score"] = np.log10(biogrid_filtered["Normalized Score"] + 1)

# Defining the quantiles
    p = np.arange(0, 1.1, 0.2)

# Assessing the quantiles of the methods
    for method in methods:
        print(method)
        print(np.quantile(biogrid_filtered[biogrid_filtered["Experimental System"] == method]["Normalized Score"], p))

# Computing the preprocessing of the BioGrid graph
    pb.create_graph()

# Plotting the distribution of scores for all methods
biogrid_filtered = pd.read_csv(f"{annot_network_dir}/biogrid_network.csv")

scores = biogrid_filtered["confidence"]
plt.figure(figsize=(10,6))
sns.histplot(scores, kde = True, bins = 30, color = "blue", alpha = 0.7)
plt.title(f"Distribution of Scores for all methods", fontsize = 14)
plt.xlabel("Score", fontsize = 12)
plt.ylabel("Frequency", fontsize = 12)
plt.grid(axis = "y", linestyle = "--", alpha = 0.7)
plt.show()

# Creating a graph only if it doesn't already exist
if not os.path.exists(f"{annot_graph_dir}/graph_BioGrid.graphml"):

# Creating an undirected graph by adding edges (iterating through the rows of the df)
    BioGridGraph = nx.Graph()

    for index, row in biogrid_filtered.iterrows():
        BioGridGraph.add_edge(row["source"], row["target"], weigth = row["confidence"])

# Saving the graph
    nx.write_graphml_lxml(BioGridGraph, f"{annot_graph_dir}/graph_BioGrid.graphml")
    
else:
    BioGridGraph = nx.read_graphml(f"{annot_graph_dir}/graph_BioGrid.graphml")

# Double-checking the graph is undirected
BioGridGraph.is_directed()

# Is the graph fully connected ?
if nx.is_connected(BioGridGraph):
    print("The graph is fully connected !")
else:
    print("The graph is not fully connected !")

# Find the connected components and their sizes
    components = list(nx.connected_components(BioGridGraph))
    components_sizes = [len(component) for component in components]
    max_components_size = max(components_sizes)

# Print the number of connected components and their sizes
    print(f"The graph has {len(components)} connected components")
    print(f"The max component size is {max_components_size}")

# Extract nodes from non-connected components (excluding the largest one)
    non_connected_components = [component for component in components if len(component) < max_components_size]

# Flatten the list of nodes
    nodes_in_non_connected_components = set().union(*non_connected_components)
    print(f"Nodes in non-connected components: {len(nodes_in_non_connected_components)}")

# Assessing how many training genes are part of non-connected components
    genes = set(genes)
    overlap_genes = nodes_in_non_connected_components.intersection(genes)
    print(f"Genes in non-connected components that are also in the training genes set= {len(overlap_genes)}")
    if len(overlap_genes) != 0:
        print(f"These genes are: {overlap_genes}")

# Setting the output folder
path_results = f"../results/{process}/BioGrid"
if not os.path.exists(path_results):
    os.mkdir(path_results)

path_results = f"../results/{process}/BioGrid/Benchmark"
if os.path.exists(f"{path_results}"):
    shutil.rmtree(f"{path_results}")
    os.mkdir(f"{path_results}")
else:
    os.mkdir(f"{path_results}")
    BGA.analyse(genes, path_results, process = process)

# Extract the largest connected component
largest_cc = max(components, key = len)
largest_cc_subgraph = BioGridGraph.subgraph(largest_cc)

# Verify the size of the largest connected component
print(f"The largest connected component has {len(largest_cc_subgraph.nodes)} nodes and {len(largest_cc_subgraph.edges)} edges")

# Extract valid genes
valid_genes = [node for node in largest_cc_subgraph.nodes(data = False) if node in genes]
print(f"{len(valid_genes)} have been selected")

# Split genes into 5 folds
kf = KFold(n_splits = 4, shuffle = True, random_state = 42)
folds = list(kf.split(valid_genes))

# Running a PageRank algorithm on the largest subgraph
all_results = []
k = 100
for df in np.arange(0.1, 1, 0.1).round(1):
    print("############## ", df, "##################")
    fold_evaluations = useful_functions.run_page_rank_databases(df, largest_cc_subgraph, folds , valid_genes , k )  # Function to run PageRank with a specific damping factor
    
 # Collect results for each fold
    for eval_result in fold_evaluations:
    # Add damping factor and fold to the result
        eval_result['damping_factor'] = df
    # Append to the results list
         all_results.append(eval_result)

    # Print the evaluation (optional)
        print(f"Fold {eval_result['fold']} Evaluation:")
        print(f"  Mean Rank: {eval_result['mean_rank']}")
        print(f"  Median Rank: {eval_result['median_rank']}")
        print(f"  Precision@{k}: {eval_result['precision_at_k']}")
        print(f"  Recall@{k}: {eval_result['recall_at_k']}")
        print(f"  AUC-ROC: {eval_result['auc_roc']}")
        print(f"  AUC-PR: {eval_result['auc_pr']}")
        print()

# Convert the results into a Pandas DataFrame
results_df = pd.DataFrame(all_results)

# Save the table to a CSV file for further analysis
results_df.to_csv(f"{path_results}/BioGrid_pagerank_largest_subgraph_{process}.csv", sep = ",", index = False)

# Averaging the results
average_results_biogrid_per_damping_factor = results_df.groupby("damping_factor").mean()

# Reset index for a cleaner display
average_results_biogrid_per_damping_factor = average_results_biogrid_per_damping_factor.reset_index()

# Save the averaged results
average_results_biogrid_per_damping_factor.to_csv(f"{path_results}/PageRank_evaluation_Biogrid_largest_subgraph_{process}_averaged.csv",
                                                  sep = ",", index = False)


# Running a PageRank algorithm on the whole graph
valid_genes = [node for node in BioGridGraph.nodes(data = False) if node in genes]

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
folds = list(kf.split(valid_genes))

all_results = []
for df in np.arange(0.1, 1, 0.1).round(1):
    print("############## ", df, "##################")
    fold_evaluations = BGA.run_page_rank_biogrid(df, BioGridGraph, k = 1000, folds = folds, valid_genes = valid_genes)  # Function to run PageRank with a specific damping factor
    
# Collect results for each fold
    for eval_result in fold_evaluations:
    # Add damping factor and fold to the result
        eval_result['damping_factor'] = df
    # Append to the results list
        all_results.append(eval_result)

# Convert the results into a Pandas DataFrame
results_df = pd.DataFrame(all_results)

# Save the table to a CSV file for further analysis
results_df.to_csv(f"{path_results}/BioGrid_pagerank_whole_graph_{process}.csv", sep = ",", index = False)

# Averaging the results
average_results_merged_per_damping_factor = results_df.groupby("damping_factor").mean()
average_results_merged_per_damping_factor = average_results_merged_per_damping_factor.reset_index()
average_results_merged_per_damping_factor

average_results_merged_per_damping_factor.to_csv(f"{path_results}/Pagerank_evaluation_BioGrid_whole_graph_{process}_averaged.csv",
                                                 sep = ",", index = False)

# Plotting the relationship between AUC and the damping factor
file = pd.read_csv(f"{path_results}/PageRank_evaluation_BioGrid_whole_graph_{process}_averaged.csv")
plt.figure(figsize = (10, 6))
plt.plot(file["damping_factor"], file["auc_roc"], marker = "o", label = "AUC-ROC", color = "blue")
plt.xlabel("Damping Factor", fontsize = 12)
plt.ylabel("AUC", fontsize = 12)
plt.title(f"BioGrid - Dependency between AUC and Damping Factor - {process.capitalize()}", fontsize = 14)
plt.grid(True, linestyle = "--", alpha = 0.6)
plt.legend(fontsize = 12)
plt.savefig(f"{path_results}/AUC_damping_factor_plot_BioGrid_{process}.png",
            format = "png", dpi = 300)

# Retrieving the best damping factor value for the biological process considered
best_df = file[file["auc_roc"] == file["auc_roc"].max()]
best_df.to_csv(f"{path_data}/Hyperparameters_BioGrid_{process}.csv", index = False, sep = ",")

### *Processing GeneCards*

<div style="color: red">
    <strong>WARNING:</strong> The original GeneCards data was obtained through a non-commercial use agreement and cannot be publicly shared.
</div>

In its original format, the GeneCards data was provided as a compressed folder containing .json files for every humain genes. The following pre-processing code extracts the necessary information from these specific files.

In [ ]:
import genecard

# Setting the process of interest
process = "stalk_cell"

# Setting the path to the original GeneCards data
path_original_GeneCards_data = "..."

# Setting the path to the GeneCards network
path_data = "../databases"

# Setting the path to the GeneCards graph
path_graph = "../graphs/databases"

# Building the network file only if it doesn't already exist
if not os.path.exists(f"{path_data}/genecards_network.csv"):

    # Creating a folder to list all the files that will be merged
    to_merge = f"{path_data}/to_merge"

    if not os.path.exists(to_merge):
        os.mkdir(to_merge)

    # Analysing the GeneCards data
    files = glob.glob(f"{path_original_GeneCards_data}/Genes/**/*.json",
                      recursive = True)

    for file in tqdm(files, desc="Processing files"):
        # Reading the input file
        df = pd.read_json(file)
        my_column_names = df.columns.tolist()

        # Retrieving the name of the gene
        gene = my_column_names[0]

        # Process interactions
        # per interactant we can have a number of details which corresponds 
        # with the number of different sources where this interaction is described
    
        source = []
        target = []
        confidence = []
        for interaction in df[gene].get("Interactions", []):
            interactant = interaction.get("Interactant")
            if interactant == "None":
                continue
        
            details_list = interaction.get("Details", [])
            processed_details = []
            if isinstance(details_list, list):
                # Process each dictionary or string in the list
                # print("The details is a list")
                for detail in details_list:
                    if isinstance(detail, dict):
                        # Convert the dictionary into a string representation
                        subdetail = detail.get("Details")
                        if (len(subdetail) == 1) & (subdetail[0] == 'IID: # pred=1'):
                            continue
                        
                        processed_details.append(str(detail))
                    else:
                        # Keep the string as is
                        print("the detail " , detail, " is not a list ")
                        processed_details.append(detail)

                # Join the processed details into a single string
                # details_str = ", ".join(processed_details) if processed_details else "No Details"
            else:
                print("The details is not a list")
                #details_str = details_list  # Handle case where "Details" is directly a string
            
            if len(processed_details) > 0: 
                source.append(gene)
                target.append(interactant)
                confidence.append(len(processed_details))
           # else:
           #     print(f"  Details: {details_str}")
           #     print(f"  Number of Details: {len(processed_details)}")
           #     print()

        if len(target) > 0:
            df_final  = pd.DataFrame({
                'source': source,
                'target': target,
                'confidence': confidence
              })
            df_final = df_final[df_final['source'] != df_final['target']]
            df_final.to_csv(f"{to_merge}/{gene}_to_merge.csv", index = False) 


    # Merging all the processed files into a global dataframe
    df = pd.concat(map(pd.read_csv, glob.glob(os.path.join(to_merge, "/*.csv"))), ignore_index = True)

    # Removing potential NAs
    df = df.dropna()

    # Removing potential self-interactions
    df = df[(df["source"] == df["target"]) == False]

    # Saving the final dataframe
    df.to_csv(f"{path_data}/network_genecards.csv", sep = ",", index = False)

else:

    # Assessing the weights in the GeneCards network
    gene_card_network = pd.read_csv(f"{path_data}/genecards_network.csv")
    gene_card_network = gene_card_network.sort_values(by = "confidence", ascending = False)
    gene_card_network = gene_card_network.drop_duplicates(subset = ["source", "target"])

    min_score = gene_card_network["confidence"].min()
    max_score = gene_card_network["confidence"].max()

    gene_card_network["weight"] = gene_card_network["confidence"]/max_score
    gene_card_network["weight"].hist()

# Reading the graph file if it already exists
if os.path.exists(f"{path_graph}/graph_GeneCards.graphml"):
    gene_card_graph = nx.read_graphml(f"{path_graph}/graph_GeneCards.graphml")

else:
# Creating a graph from scratch
    gene_card_graph = nx.from_pandas_edgelist(gene_card_network,
                                              source = "source",
                                              target = "target",
                                              edge_attr = "weight",
                                              create_using = nx.Graph())

# Saving the graph
    nx.write_graphml(gene_card_graph, f"{path_graph}/graph_GeneCards.graphml")

# Setting the path to the results
path_results = f"../results/{process}/GeneCards"

if not os.path.exists(path_results):
    os.mkdir(path_results)

path_results = f"../results/{process}/GeneCards/Benchmark"
    
if os.path.exists(f"{path_results}"):
    shutil.rmtree(f"{path_results}")
    os.mkdir(f"{path_results}")
        
else:
    os.mkdir(f"{path_results}")
     
# Running a PageRank algorithm on the whole graph
valid_genes = [node for node in gene_card_graph.nodes(data = False) if node in genes]

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
folds = list(kf.split(valid_genes))
print(f"There are {len(valid_genes)} valid genes")

all_results = []
k = 1000
for df in np.arange(0.1,1, 0.1).round(1):
    print("############## ", df, "##################")
    fold_evaluations = genecard.run_page_rank_genecard(df, gene_card_graph, k, folds, valid_genes)  # Function to run PageRank with a specific damping factor
    
# Collect results for each fold
    for eval_result in fold_evaluations:
    # Add damping factor and fold to the result
        eval_result['damping_factor'] = df
    # Append to the results list
        all_results.append(eval_result)

    # Print the evaluation (optional)
        print(f"Fold {eval_result['fold']} Evaluation:")
        print(f"  Mean Rank: {eval_result['mean_rank']}")
        print(f"  Median Rank: {eval_result['median_rank']}")
        print(f"  Precision@{k}: {eval_result['precision_at_k']}")
        print(f"  Recall@{k}: {eval_result['recall_at_k']}")
        print(f"  AUC-ROC: {eval_result['auc_roc']}")
        print(f"  AUC-PR: {eval_result['auc_pr']}")
        print()

# Convert the results into a Pandas DataFrame
results_df = pd.DataFrame(all_results)

# Save the table to a CSV file for further analysis
results_df.to_csv(f"{path_results}/GeneCards_pagerank_whole_graph_{process}.csv", sep = ",", index = False)

# Averaging the results 
average_results_gene_card_per_damping_factor = results_df.groupby("damping_factor").mean()

# Reset index for cleaner display
average_results_gene_card_per_damping_factor = average_results_gene_card_per_damping_factor.reset_index()

# Saving the average results
average_results_gene_card_per_damping_factor.to_csv(f"{path_results}/PageRank_evaluation_GeneCards_whole_graph_{process}_averaged.csv", 
                                                    sep = ",", index = False)
    
# Retrieving the best damping factor for the biological process considered
best_df = average_results_gene_card_per_damping_factor[average_results_gene_card_per_damping_factor["auc_roc"] == average_results_gene_card_per_damping_factor["auc_roc"].max()]
best_df.to_csv(f"{path_data}/Hyperparameters_GeneCards_{process}.csv", index = False, sep = ",")

### *Processing ConsensusPath*

In [ ]:
import consensus

# Setting the process of interest
process = "stalk_cell"

# Setting the path to the ConsensusPath data
path_data = "../databases"

# Setting the path to the ConsensusPath graph
path_graph = "../graphs/databases"

# Creating the network file only if it doesn't already exist:
if not os.path.exists(f"{path_data}/consensuspath_network.csv"):

# Reading the original data
    consensus_network = pd.read_csv(f"{path_data}/ConsensusPathDB_human_PPI", sep = "\t") # /!\ Manually delete the first row in the original data file /!\

# Extracting a subset with relevant columns only
    cols = ["interaction_participants__genename",
            "interaction_confidence",
            "interaction_publications"]

    consensus_network = consensus_network[cols]

# Adding a column giving the number of publications per interaction
    consensus_network["publication_count"] = consensus_network["interaction_publications"].apply(lambda x: len(x.split(',')) if isinstance(x, str) else 0)

# Plotting the interaction confidence scores
    consensus_network.hist("interaction_confidence")

# Adding a column giving the number of interactor per interaction
    consensus_network["n_interactors"] = consensus_network["interaction_participants__genename"].apply(lambda x: len(x.split(',')) if isinstance(x, str) else 0)

# Calculate the average interaction_confidence for each publication_count group
    average_confidence_by_publication_count = (
        consensus_network.groupby("publication_count")["interaction_confidence"]
        .mean()
        .to_dict()
    )

# Replace NaN values in interaction_confidence
    consensus_network["interaction_confidence"] = consensus_network.apply(
        lambda row: average_confidence_by_publication_count[row["publication_count"]]
        if pd.isna(row["interaction_confidence"])
        else row["interaction_confidence"],
        axis=1
    )

# Removing NAs
    consensus_network = consensus_network.dropna()
    
# Applying the create_binary_interactions function for each row
    consensus_network["binary_interactions"] = consensus_network["interaction_participants__genename"].apply(consensus.create_binary_interactions)

# Explode the binary_interaction column to get one pair per row
    binary_df = consensus_network.explode("binary_interactions").reset_index(drop = True)

# Removing NAs
    binary_df = binary_df.dropna()

# Convert tuples into readable string format
    binary_df["binary_interactions"] = binary_df["binary_interactions"].apply(lambda x: ",".join(x))

# Plotting the interaction confidence vs the publication count
    plt.figure(figsize=(8, 6))
    plt.scatter(np.log(consensus_network["publication_count"]), consensus_network["interaction_confidence"],
                color = "blue", alpha = 0.7)

    plt.title("Interaction Confidence vs Publication Count", fontsize = 14)
    plt.xlabel("Publication Count", fontsize = 12)
    plt.ylabel("Interaction Confidence", fontsize = 12)

    plt.grid(alpha =  0.3)

    plt.savefig(f"{path_data}/interaction_confidence_vs_publication_count.png")
    plt.show()

# Splitting the interactions to make them readable
    binary_df[["source", "target"]] = binary_df["binary_interactions"].str.split(",", expand = True)

# Renaming the "confidence" column to "weight"
    binary_df = binary_df.rename(columns = {"interaction_confidence": "weight"})

# Keeping only the required clumns
    binary_df = binary_df[["source", "target", "weight"]]

# Removing self-interactions
    binary_df = binary_df[binary_df["source"] != binary_df["target"]]

# Saving the final network file
    binary_df.to_csv(f"{path_data}/consensuspath_network.csv", sep = ",", index = False)

else:
    binary_df = pd.read_csv(f"{path_data}/consensuspath_network.csv")
    

# Creating a graph from the network file
if not os.path.exists(f"{path_graph}/graph_ConsensusPath.graphml"):
    consensus_graph = consensus.create_graph(f"{path_data}/consensuspath_network.csv")
    nx.write_graphml(consensus_graph, f"{path_graph}/graph_ConsensusPath.graphml")

else:
    consensus_graph = nx.read_graphml(f"{path_graph}/graph_ConsensusPath.graphml")

  
# Setting the path to the results
path_results = f"../results/{process}/ConsensusPath"

if not os.path.exists(f"{path_results}"):
    os.mkdir(f"{path_results}")

path_results = f"../results/{process}/ConsensusPath/Benchmark"

if not os.path.exists(path_results):
    os.mkdir(path_results)
else:
    shutil.rmtree(path_results)
    os.mkdir(path_results)

# Analysing the connectivity of the training genes in the network
consensus.analyse(genes, consensus_graph, path_results)

# Running a PageRank algorithm on the whole graph
valid_genes = [node for node in consensus_graph.nodes(data = False) if node in genes]

kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
folds = list(kf.split(valid_genes))
print(f"There are {len(valid_genes)} valid genes")

all_results = []
k = 1000
for df in np.arange(0.1,1, 0.1).round(1):
    print("############## ", df, "##################")
    fold_evaluations = consensus.run_page_rank_consensus(df, consensus_graph, k, folds, valid_genes)  # Function to run PageRank with a specific damping factor
    
# Collect results for each fold
    for eval_result in fold_evaluations:
    # Add damping factor and fold to the result
        eval_result['damping_factor'] = df
    # Append to the results list
        all_results.append(eval_result)

    # Print the evaluation (optional)
        print(f"Fold {eval_result['fold']} Evaluation:")
        print(f"  Mean Rank: {eval_result['mean_rank']}")
        print(f"  Median Rank: {eval_result['median_rank']}")
        print(f"  Precision@{k}: {eval_result['precision_at_k']}")
        print(f"  Recall@{k}: {eval_result['recall_at_k']}")
        print(f"  AUC-ROC: {eval_result['auc_roc']}")
        print(f"  AUC-PR: {eval_result['auc_pr']}")
        print()

# Convert the results into a Pandas DataFrame
results_df = pd.DataFrame(all_results)

# Save the table to a CSV file for further analysis
results_df.to_csv(f"{path_results}/ConsensusPath_pagerank_whole_graph.csv", sep = ",", index = False)

# Averaging the results 
average_results_consensus_per_damping_factor = results_df.groupby("damping_factor").mean()

# Reset index for cleaner display
average_results_consensus_per_damping_factor = average_results_consensus_per_damping_factor.reset_index()

# Saving the average results
average_results_consensus_per_damping_factor.to_csv(f"{path_results}/PageRank_evaluation_ConsensusPath_whole_graph_{process}.csv", 
                                                    sep = ",", index = False)

# Retrieving the best damping factor value for the biological process considered
best_df = average_results_consensus_per_damping_factor[average_results_consensus_per_damping_factor["auc_roc"] == average_results_consensus_per_damping_factor["auc_roc"].max()]
best_df.to_csv(f"{path_data}/Hyperparameters_ConsensusPath_{process}.csv", index = False, sep = ",")

### *Processing STRING*

In [ ]:
import string_analysis

# Setting the process of interest
process = "stalk_cell"

# Setting the path to the STRING data
path_data = "../databases"

# Setting the path to the ConsensusPath graph
path_graph = "../graphs/databases"

# Creating the STRING network file only if it doesn't exist
if not os.path.exists(f"{path_data}/string_network.csv"):

    # Reading the raw input files
    info_file = f"{path_data}/9606.protein.info.v12.0.txt"
    link_file = f"{path_data}/9606.protein.links.v12.0.txt"

    string_network = pd.read_csv(f"{link_file}", sep = " ")
    protein_info = pd.read_csv(f"{info_file}", sep = "\t")

    # Creating a dictionary to map ensembl ids to symbol ids
    id_to_name = dict(zip(protein_info["#string_protein_id"], protein_info["preferred_name"]))

    # Add gene names to the network data
    string_network["gene1"] = string_network["protein1"].map(id_to_name)
    string_network["gene2"] = string_network["protein2"].map(id_to_name)

    # Remove rows where mapping is not possible (e.g. missing genes)
    string_network = string_network.dropna(subset = ["gene1", "gene2"])

    # Remove self-interactions
    string_network = string_network[string_network["gene1"] != string_network["gene2"]]

    # Sort the values by descending combined score
    string_network = string_network.sort_values(by = "combined_score", ascending = False)

    # Drop duplicates
    string_network = string_network.drop_duplicates(subset = ["protein1", "protein2"])

    # Saving the network
    string_network.to_csv(f"{path_data}/string_network.csv", sep = ",", index = False)

        
# Generating the STRING graphs for the specific biological process of interest
if not os.path.exists(f"{path_graph}/graph_STRING_{process}.graphml"):
    path_results = f"../results/{process}/STRING"
    if not os.path.exists(path_results):
        os.mkdir(path_results)

# Create a folder for the upcoming threshold benchmark
    path_results = f"../results/{process}/STRING/Benchmark_thresholds"
    if not os.path.exists(path_results):
        os.mkdir(path_results)

# Reading the network file
    string_network = pd.read_csv(f"{path_data}/network_STRING.csv")
       
    # Running a benchmark on the combined_score values to filter the data
    # and find the best threshold for gene prioritization
    thresholds = [100, 200, 300, 400, 500, 600, 700]
    for threshold in tqdm(thresholds, position = 0, desc = "Processing thresholds ..."):
        print(f"Processing graph for threshold {threshold} and process {process}")

        # Initiating the workflow
        string_graph = string_analysis.make_string_graph(string_network, threshold)
        n_nodes = len(string_graph.nodes())
        n_edges = len(string_graph.edges())
        valid_genes = [node for node in string_graph.nodes(data = False) if node in genes]

        kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
        folds = list(kf.split(valid_genes))

        all_results = []
        k = 100
        for df in np.arange(0.1, 1, 0.1).round(1):
            print("########## Processing damping factor ", df, "##########")
            fold_evaluations = useful_functions.run_page_rank_databases(df, 
                                                                        string_graph, 
                                                                        folds, 
                                                                        valid_genes, 
                                                                        k)

            # Retrieving the results for each fold
            for eval_result in fold_evaluations:
                # Add damping factor and fold to the results
                eval_result["damping_factor"] = df
                # Append this to the results list
                all_results.append(eval_result)

        results_df = pd.DataFrame(all_results)
        average_results_string_per_damping_factor = results_df.groupby("damping_factor").mean()
        average_results_string_per_damping_factor["n_nodes"] = n_nodes
        average_results_string_per_damping_factor["n_edges"] = n_edges
        average_results_string_per_damping_factor["n_tr_genes"] = len(valid_genes)
        average_results_string_per_damping_factor["threshold"] = threshold

        # Reset index for a cleaner display
        average_results_string_per_damping_factor = average_results_string_per_damping_factor.reset_index()

        # Saving the results
        average_results_string_per_damping_factor.to_csv(f"{path_results}/pagerank_evaluations_string_graph_{threshold}_{process}.csv",
                                                         sep = ",", index = False)

    # Analyzing the benchmark results : we need to determine the best threshold for the scores and the best damping factor
    thresholds = []
    AUCs = []
    Damping_factors = []

    files = glob.glob(f"{path_results}/*.csv")
    for file in files:
        results = pd.read_csv(file)
        thresholds.append(int(results.loc[:, "threshold"].mean()))

        best_df = results[results["auc_roc"] == results["auc_roc"].max()]
        for i in best_df.itertuples():
            Damping_factors.append(i[1])
            AUCs.append(i[7])

    benchmark_process = pd.DataFrame({"Threshold": thresholds,
                                      "Best_AUC": AUCs,
                                      "Best_DF": Damping_factors})

    benchmark_process.to_csv(f"{path_results}/Benchmark_thresholds_STRING_{process}.csv",
                             sep = ",", index = False)

    # Using the benchmark to find the hyperparameters (scores threshold and damping factor)
    hyperparameters_df = benchmark_process[benchmark_process["Best_AUC"] == benchmark_process["Best_AUC"].max()]
    hyperparameters_df.to_csv(f"{path_data}/Hyperparameters_STRING_{process}.csv", sep = ",", index = False)

    # Using the best threshold to create a graph file
    for i in hyperparameters_df.itertuples():
        best_threshold = i[1]

    string_graph = string_analysis.make_string_graph(string_network, threshold)
    nx.write_graphml(string_graph, f"{path_graph}/graph_STRING_{process}.graphml")